# Financial Analytics — Group Project
## Tickers: ABNB · CCL · TSLA · KER.PA · PUM.DE | 2023–2025
**Sections:** EDA → Distribution Analysis → GARCH Volatility → VaR / ES → Portfolio Optimization → Q1 2026 Evaluation

> Run cells top to bottom. All figures are saved to `figures/` for copy-paste into slides.

In [ ]:
# ── INSTALL DEPENDENCIES (Google Colab — run this cell first, then restart runtime) ──
!pip install arch cvxpy -q
# yfinance, seaborn, scipy, numpy, pandas, matplotlib are already in Colab

In [ ]:
# ── 0. SETUP ──────────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from scipy.stats import norm
from arch import arch_model
import cvxpy as cp
import os

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': '#F8F9FA',
    'axes.grid': True,
    'grid.alpha': 0.35,
    'font.size': 10,
    'axes.titlesize': 11,
    'axes.titleweight': 'bold'
})

# ── Constants ─────────────────────────────────────────────────────────────────
TICKERS = ['ABNB', 'CCL', 'TSLA', 'KER.PA', 'PUM.DE']
NAMES   = {'ABNB': 'Airbnb', 'CCL': 'Carnival',
           'TSLA': 'Tesla', 'KER.PA': 'Kering', 'PUM.DE': 'Puma'}
COLORS  = {'ABNB': '#E63946', 'CCL': '#2196F3', 'TSLA': '#FF6F00',
           'KER.PA': '#9C27B0', 'PUM.DE': '#4CAF50'}
START, END   = '2023-01-01', '2025-12-31'
RF_MONTHLY   = 0.004       # 0.4 % monthly ≈ 5 % annual (US T-bill 2023-25)
HOLDING      = 21          # 21 trading days ≈ 1 month
SAVE_DPI     = 150

os.makedirs('figures', exist_ok=True)
print('Setup complete — figures/ directory ready.')

In [ ]:
# ── 1. DATA DOWNLOAD ──────────────────────────────────────────────────────────
import time

def _download_one(ticker, start, end, retries=4, delay=3):
    """Download a single ticker with retries to handle Colab SQLite cache locks."""
    for attempt in range(retries):
        try:
            df = yf.download(ticker, start=start, end=end,
                             auto_adjust=False, progress=False)
            if len(df) > 10:                      # sanity: got real data
                px = df['Close']
                if isinstance(px, pd.DataFrame):  # flatten if multi-col
                    px = px.iloc[:, 0]
                px.name = ticker
                return px
            print(f'  {ticker}: empty response, retrying ({attempt+1}/{retries})…')
        except Exception as e:
            print(f'  {ticker}: error "{e}", retrying ({attempt+1}/{retries})…')
        time.sleep(delay)
    raise RuntimeError(f'Could not download {ticker} after {retries} attempts.')

print('Downloading prices (one ticker at a time to avoid Colab cache lock)…')
all_tickers = TICKERS + ['^GSPC']
px_dict = {}
for t in all_tickers:
    print(f'  {t}…', end=' ', flush=True)
    px_dict[t] = _download_one(t, START, END)
    print(f'✓  {len(px_dict[t])} days')

# Combine into a single DataFrame
prices  = pd.DataFrame({t: px_dict[t] for t in TICKERS})
mkt_px  = px_dict['^GSPC']

# Log-returns
log_ret = np.log(prices / prices.shift(1))
mkt_ret = np.log(mkt_px / mkt_px.shift(1))

# Align to common trading days (inner join, drop any remaining NaNs)
all_data = pd.concat([log_ret, mkt_ret.rename('^GSPC')], axis=1).dropna()
log_ret  = all_data[TICKERS]
mkt_ret  = all_data['^GSPC']

print(f'\nPeriod : {log_ret.index[0].date()} to {log_ret.index[-1].date()}')
print(f'Days   : {len(log_ret)} common trading-day observations')
log_ret.head(3)

---
## Section 1 — Exploratory Data Analysis

In [ ]:
# ── 1A. NORMALIZED PRICE SERIES ───────────────────────────────────────────────
norm_px = prices.divide(prices.bfill().iloc[0]) * 100   # rebase to 100

fig, axes = plt.subplots(1, 5, figsize=(18, 4), sharey=False)
for i, t in enumerate(TICKERS):
    ax = axes[i]
    ax.plot(norm_px.index, norm_px[t], color=COLORS[t], linewidth=1.5)
    ax.axhline(100, color='gray', linestyle='--', linewidth=0.8, alpha=0.7)
    ax.set_title(NAMES[t])
    ax.tick_params(axis='x', rotation=40, labelsize=8)
    if i == 0:
        ax.set_ylabel('Index (Jan 2023 = 100)')

fig.suptitle('Normalized Price Series — Jan 2023 to Dec 2025', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('figures/01_normalized_prices.png', dpi=SAVE_DPI, bbox_inches='tight')
plt.show()
print('Saved → figures/01_normalized_prices.png')

In [ ]:
# ── 1B. DAILY LOG-RETURNS TIME SERIES ─────────────────────────────────────────
fig, axes = plt.subplots(5, 1, figsize=(14, 12), sharex=True)
for i, t in enumerate(TICKERS):
    ax = axes[i]
    ax.plot(log_ret.index, log_ret[t] * 100, color=COLORS[t], linewidth=0.75, alpha=0.85)
    ax.axhline(0, color='black', linewidth=0.5)
    ax.set_ylabel(f'{NAMES[t]}\n(%)', fontsize=9)

axes[0].set_title('Daily Log-Returns (%) — All Tickers, 2023–2025')
axes[-1].set_xlabel('Date')
plt.tight_layout()
plt.savefig('figures/02_log_returns.png', dpi=SAVE_DPI, bbox_inches='tight')
plt.show()
print('Saved → figures/02_log_returns.png')

In [ ]:
# ── 1C. SUMMARY STATISTICS ────────────────────────────────────────────────────
from scipy.stats import skew, kurtosis as excess_kurt

stats_rows = []
for t in TICKERS:
    r = log_ret[t].dropna().values
    stats_rows.append({
        'Name'         : NAMES[t],
        'Mean (daily%)': round(r.mean() * 100, 4),
        'Std (daily%)' : round(r.std()  * 100, 4),
        'Ann. Vol (%)' : round(r.std()  * np.sqrt(252) * 100, 2),
        'Skewness'     : round(skew(r), 3),
        'Ex. Kurtosis' : round(excess_kurt(r), 3),
        'Min (%)'      : round(r.min() * 100, 3),
        'Max (%)'      : round(r.max() * 100, 3),
    })

stats_df = pd.DataFrame(stats_rows).set_index('Name')
print('=== Summary Statistics of Daily Log-Returns ===')
print(stats_df.to_string())

# Save for reference
stats_df.to_csv('figures/summary_statistics.csv')
print('\nSaved → figures/summary_statistics.csv')

In [ ]:
# ── 1D. ROLLING 30-DAY ANNUALISED VOLATILITY ──────────────────────────────────
rolling_vol = log_ret.rolling(30).std() * np.sqrt(252) * 100   # in %

fig, ax = plt.subplots(figsize=(14, 5))
for t in TICKERS:
    ax.plot(rolling_vol.index, rolling_vol[t],
            label=NAMES[t], color=COLORS[t], linewidth=1.6)
ax.set_title('30-Day Rolling Annualised Volatility (%) — 2023–2025')
ax.set_ylabel('Annualised Volatility (%)')
ax.set_xlabel('Date')
ax.legend(ncol=5, loc='upper right', fontsize=9)
plt.tight_layout()
plt.savefig('figures/03_rolling_volatility.png', dpi=SAVE_DPI, bbox_inches='tight')
plt.show()
print('Saved → figures/03_rolling_volatility.png')

---
## Section 2 — Distribution Analysis (Normal vs Student-t)

In [ ]:
# ── 2. NORMAL vs STUDENT-t FIT + KS TESTS ────────────────────────────────────
print('=== Distribution Fitting & KS Tests ===\n')
dist_results = {}
for t in TICKERS:
    x = log_ret[t].dropna().values
    mu_n, sig_n       = stats.norm.fit(x)
    df_t, loc_t, sc_t = stats.t.fit(x)
    ks_n = stats.kstest(x, 'norm', args=(mu_n, sig_n))
    ks_t = stats.kstest(x, 't',    args=(df_t, loc_t, sc_t))
    dist_results[t] = {'df': df_t, 'loc': loc_t, 'scale': sc_t,
                       'ks_norm_p': ks_n.pvalue, 'ks_t_p': ks_t.pvalue}
    print(f'{NAMES[t]:20s} | t-df = {df_t:5.2f} | '
          f'KS-Normal p = {ks_n.pvalue:.4f} | KS-t p = {ks_t.pvalue:.4f}')

print('\n→ Low p-value (< 0.05) for Normal confirms non-normality.')
print('→ t-df < 5 signals heavy tails / frequent extreme returns.')

# Bar chart of t degrees-of-freedom per ticker
fig, ax = plt.subplots(figsize=(8, 4))
dfs = [dist_results[t]['df'] for t in TICKERS]
bars = ax.bar([NAMES[t] for t in TICKERS], dfs,
              color=[COLORS[t] for t in TICKERS], edgecolor='black', linewidth=0.5)
ax.axhline(5, color='gray', linestyle='--', linewidth=1, label='df = 5 (moderate tails)')
for b, v in zip(bars, dfs):
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.05,
            f'{v:.1f}', ha='center', va='bottom', fontsize=9)
ax.set_title('Student-t Degrees of Freedom per Ticker\n(lower = heavier tails)')
ax.set_ylabel('Estimated df')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('figures/04_t_degrees_of_freedom.png', dpi=SAVE_DPI, bbox_inches='tight')
plt.show()
print('Saved → figures/04_t_degrees_of_freedom.png')

---
## Section 3 — Volatility Modelling: GARCH vs GJR-GARCH

In [ ]:
# ── 3A. MODEL SELECTION: GARCH(1,1)-t  vs  GJR-GARCH(1,1,1)-t ────────────────
print('=== GARCH Model Selection (by AIC) ===\n')
garch_res = {}

for t in TICKERS:
    ret_pct = (100 * log_ret[t]).dropna()

    fg = arch_model(ret_pct, vol='GARCH', p=1, q=1,
                    mean='Constant', dist='t').fit(disp='off')
    fj = arch_model(ret_pct, vol='GARCH', p=1, o=1, q=1,
                    mean='Constant', dist='t').fit(disp='off')

    best      = fg if fg.aic <= fj.aic else fj
    best_name = 'GARCH(1,1)-t' if fg.aic <= fj.aic else 'GJR-GARCH-t'

    p = best.params
    alpha = p.get('alpha[1]', 0)
    beta  = p.get('beta[1]',  0)
    gamma = p.get('gamma[1]', 0)   # 0 if plain GARCH
    persist = alpha + beta + 0.5 * gamma

    garch_res[t] = {
        'fit'     : best,
        'model'   : best_name,
        'aic_g'   : fg.aic,
        'aic_gjr' : fj.aic,
        'persist' : persist,
        'params'  : p
    }
    print(f'{NAMES[t]:20s} | Winner: {best_name:15s} | AIC = {best.aic:.1f} | '
          f'Persistence = {persist:.4f}')

print('\n→ Persistence close to 1 means volatility shocks decay very slowly.')
print('→ GJR winning signals a leverage effect: negative news → more volatility.')

In [ ]:
# ── 3B. CONDITIONAL VOLATILITY PLOTS ──────────────────────────────────────────
fig, axes = plt.subplots(5, 1, figsize=(14, 13), sharex=True)
for i, t in enumerate(TICKERS):
    ax      = axes[i]
    ret_pct = (100 * log_ret[t]).dropna()
    cv      = garch_res[t]['fit'].conditional_volatility

    ax.plot(ret_pct.index, ret_pct.values,
            color='gray', linewidth=0.65, alpha=0.55, label='Return (%)')
    ax.fill_between(ret_pct.index, -2*cv, 2*cv,
                    color=COLORS[t], alpha=0.28, label=f'±2σ ({garch_res[t]["model"]})')
    ax.set_ylabel(NAMES[t], fontsize=9)
    ax.legend(loc='upper right', fontsize=7, ncol=2)

axes[0].set_title('Log-Returns with GARCH Conditional Volatility Band (±2σ), 2023–2025')
axes[-1].set_xlabel('Date')
plt.tight_layout()
plt.savefig('figures/05_conditional_volatility.png', dpi=SAVE_DPI, bbox_inches='tight')
plt.show()
print('Saved → figures/05_conditional_volatility.png')

---
## Section 4 — Risk Measures: VaR and Expected Shortfall

In [ ]:
# ── 4A. VaR / ES FUNCTIONS (from Lecture 8) ───────────────────────────────────
def compute_var(losses, levels):
    """losses: positive daily loss series in %. Returns DataFrame."""
    x = pd.Series(losses).dropna().astype(float)
    df_t, loc_t, sc_t = stats.t.fit(x)
    mu, sig = x.mean(), x.std(ddof=1)
    rows = []
    for a in levels:
        rows.append({
            'Level'      : a,
            'Historical' : np.percentile(x, 100*a),
            'Normal'     : mu + sig * norm.ppf(a),
            'T-dist'     : stats.t.ppf(a, df_t, loc=loc_t, scale=sc_t)
        })
    return pd.DataFrame(rows).set_index('Level')

def compute_es(losses, levels):
    """losses: positive daily loss series in %. Returns DataFrame."""
    x = pd.Series(losses).dropna().astype(float)
    df_t, loc_t, sc_t = stats.t.fit(x)
    mu, sig = x.mean(), x.std(ddof=1)
    rows = []
    for a in levels:
        var_h = np.percentile(x, 100*a)
        es_h  = x[x >= var_h].mean()
        z     = norm.ppf(a)
        es_n  = mu + sig * norm.pdf(z) / (1 - a)
        q     = stats.t.ppf(a, df_t)
        es_t  = (loc_t + sc_t * (df_t + q**2) / ((df_t - 1) * (1 - a))
                 * stats.t.pdf(q, df_t)) if df_t > 1 else np.nan
        rows.append({'Level': a, 'Historical': es_h, 'Normal': es_n, 'T-dist': es_t})
    return pd.DataFrame(rows).set_index('Level')

print('VaR / ES functions ready.')

In [ ]:
# ── 4B. COMPUTE AND COMPARE VaR / ES ACROSS TICKERS ──────────────────────────
LEVELS = [0.95, 0.99]
var_all, es_all = {}, {}

for t in TICKERS:
    loss = -(100 * log_ret[t]).dropna()
    var_all[t] = compute_var(loss, LEVELS)
    es_all[t]  = compute_es(loss, LEVELS)

# Print comparison tables
for a in LEVELS:
    print(f'\n=== VaR at {a:.0%} (daily loss %) ===')
    rows = [{'Name': NAMES[t],
             'Historical': round(var_all[t].loc[a, 'Historical'], 3),
             'Normal'    : round(var_all[t].loc[a, 'Normal'],     3),
             'T-dist'    : round(var_all[t].loc[a, 'T-dist'],     3)}
            for t in TICKERS]
    print(pd.DataFrame(rows).set_index('Name').to_string())

for a in LEVELS:
    print(f'\n=== ES at {a:.0%} (daily loss %) ===')
    rows = [{'Name': NAMES[t],
             'Historical': round(es_all[t].loc[a, 'Historical'], 3),
             'Normal'    : round(es_all[t].loc[a, 'Normal'],     3),
             'T-dist'    : round(es_all[t].loc[a, 'T-dist'],     3)}
            for t in TICKERS]
    print(pd.DataFrame(rows).set_index('Name').to_string())

# ── Bar chart: VaR 99% and ES 99% (T-dist) ───────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
labels = [NAMES[t] for t in TICKERS]
colors = [COLORS[t] for t in TICKERS]
var99  = [var_all[t].loc[0.99, 'T-dist'] for t in TICKERS]
es99   = [es_all[t].loc[0.99,  'T-dist'] for t in TICKERS]

for ax, vals, title, ylabel in [
        (axes[0], var99, 'Value-at-Risk 99% — T-distribution (daily loss %)',  'Loss (%)'),
        (axes[1], es99,  'Expected Shortfall 99% — T-distribution (daily loss %)', 'Loss (%)')]:
    bars = ax.bar(labels, vals, color=colors, edgecolor='black', linewidth=0.5)
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.02,
                f'{v:.2f}%', ha='center', va='bottom', fontsize=9)
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.savefig('figures/06_var_es_comparison.png', dpi=SAVE_DPI, bbox_inches='tight')
plt.show()
print('Saved → figures/06_var_es_comparison.png')

In [ ]:
# ── 4C. DYNAMIC VaR — most volatile ticker (TSLA) ────────────────────────────
# Reference: Code_Lecture_8.ipynb, cells 15-18
t_dyn = 'TSLA'
ret_pct_dyn = (100 * log_ret[t_dyn]).dropna()

mod_dyn = arch_model(ret_pct_dyn, vol='Garch', p=1, q=1,
                     mean='AR', lags=1, dist='t')
res_dyn = mod_dyn.fit(disp='off')

fcst    = res_dyn.forecast(start=0)
mu_r    = fcst.mean.iloc[:, -1]
sig_r   = np.sqrt(fcst.variance.iloc[:, -1])
nu      = res_dyn.params['nu']
lam     = sig_r * np.sqrt((nu - 2) / nu)

idx = mu_r.index
q95 = stats.t.ppf(0.05, df=nu, loc=mu_r.values, scale=lam.values)
q99 = stats.t.ppf(0.01, df=nu, loc=mu_r.values, scale=lam.values)

var95_dyn = pd.Series(-q95, index=idx)
var99_dyn = pd.Series(-q99, index=idx)
loss_dyn  = (-ret_pct_dyn).reindex(idx)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(loss_dyn.index, loss_dyn, color='gray', alpha=0.5, linewidth=0.8, label='Loss (%)')
ax.plot(var99_dyn.index, var99_dyn, color='#B71C1C', linestyle='--', linewidth=1.5, label='Dynamic VaR 99%')
ax.plot(var95_dyn.index, var95_dyn, color='#E57373', linestyle='--', linewidth=1.2, label='Dynamic VaR 95%')
ax.axhline(0, color='black', linewidth=0.4)
ax.set_title(f'Dynamic VaR — {NAMES[t_dyn]} (AR(1)-GARCH(1,1)-t, 2023–2025)')
ax.set_xlabel('Date')
ax.set_ylabel('Daily Loss (%)')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('figures/07_dynamic_var_tsla.png', dpi=SAVE_DPI, bbox_inches='tight')
plt.show()
print('Saved → figures/07_dynamic_var_tsla.png')

---
## Section 6 — Portfolio Analysis

In [ ]:
# ── 6A. CORRELATION HEATMAP ────────────────────────────────────────────────────
corr = log_ret.corr()
names_list = [NAMES[t] for t in TICKERS]
corr_named = corr.rename(index=NAMES, columns=NAMES)

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(corr_named, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, ax=ax,
            linewidths=0.5, annot_kws={'size': 11},
            cbar_kws={'shrink': 0.8})
ax.set_title('Return Correlation Matrix — 2023–2025')
plt.tight_layout()
plt.savefig('figures/09_correlation_heatmap.png', dpi=SAVE_DPI, bbox_inches='tight')
plt.show()
print('Saved → figures/09_correlation_heatmap.png')
print('\nCorrelation Matrix:')
print(corr_named.round(3))

In [ ]:
# ── 6B. EFFICIENT FRONTIER FUNCTION (from Lecture 11) ─────────────────────────
def compute_frontier(rf, mean_ret, cov_mat, n_pts=80):
    """
    Long-only efficient frontier + tangency portfolio via cvxpy.
    Returns dict with 'risks', 'returns', 'min_var', 'tangency', 'portfolios'.
    """
    targets   = np.linspace(mean_ret.min(), mean_ret.max(), n_pts)
    risks, rets, weights, sharpes = [], [], [], []
    mv_r, mv_ret, mv_w = np.inf, None, None

    Sigma = cov_mat.values.astype(float)
    mu    = mean_ret.values.astype(float)
    n     = len(mu)
    # Small regularisation for numerical stability
    Sigma_reg = Sigma + 1e-8 * np.eye(n)

    for tgt in targets:
        w = cp.Variable(n)
        obj  = cp.Minimize(cp.quad_form(w, Sigma_reg))
        cons = [cp.sum(w) == 1, w >= 0, mu @ w == tgt]
        prob = cp.Problem(obj, cons)
        prob.solve(solver=cp.SCS, verbose=False, eps=1e-5)

        if prob.status not in ('optimal', 'optimal_inaccurate') or w.value is None:
            continue

        wv = np.clip(w.value, 0, None)
        wv /= wv.sum()                            # re-normalise numerical noise
        pvar = float(wv @ Sigma @ wv)
        pstd = np.sqrt(max(pvar, 0))
        pret = float(mu @ wv)

        risks.append(pstd); rets.append(pret)
        weights.append(wv)
        sharpes.append((pret - rf) / pstd if pstd > 0 else -np.inf)

        if pstd < mv_r:
            mv_r, mv_ret, mv_w = pstd, pret, wv

    idx_tang = int(np.argmax(sharpes))
    return {
        'risks'     : np.array(risks),
        'returns'   : np.array(rets),
        'portfolios': weights,
        'sharpes'   : np.array(sharpes),
        'min_var'   : (mv_r, mv_ret, mv_w),
        'tangency'  : (risks[idx_tang], rets[idx_tang],
                       weights[idx_tang], sharpes[idx_tang])
    }

print('Frontier function loaded.')

In [ ]:
# ── 6C. RUN PORTFOLIO OPTIMISATION ────────────────────────────────────────────
mean_21d = log_ret.mean() * HOLDING
cov_21d  = log_ret.cov()  * HOLDING
std_21d  = log_ret.std()  * np.sqrt(HOLDING)

print('Running efficient frontier optimisation (≈ 10 s) …')
frontier = compute_frontier(RF_MONTHLY, mean_21d, cov_21d)

mv_r,   mv_ret,   mv_w            = frontier['min_var']
tg_r,   tg_ret,   tg_w, tg_sr    = frontier['tangency']

print('\n=== Minimum Variance Portfolio (21-day) ===')
for t, w in zip(TICKERS, mv_w):
    print(f'  {NAMES[t]:20s}: {w*100:.1f}%')
print(f'  Expected Return : {mv_ret*100:.3f}%')
print(f'  Std Deviation   : {mv_r*100:.3f}%')

print(f'\n=== Tangency Portfolio (Sharpe = {tg_sr:.3f}, 21-day) ===')
for t, w in zip(TICKERS, tg_w):
    print(f'  {NAMES[t]:20s}: {w*100:.1f}%')
print(f'  Expected Return : {tg_ret*100:.3f}%')
print(f'  Std Deviation   : {tg_r*100:.3f}%')

In [ ]:
# ── 6D. EFFICIENT FRONTIER PLOT + CAL ─────────────────────────────────────────
risks_f = frontier['risks']
rets_f  = frontier['returns']
mv_idx  = int(np.argmin(risks_f))

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# ── Left: Frontier ─────────────────────────────────────────────────────────
ax = axes[0]
ax.plot(risks_f[:mv_idx+1], rets_f[:mv_idx+1],
        color='gold', linewidth=2, linestyle='--', label='Inefficient portfolios')
ax.plot(risks_f[mv_idx:],   rets_f[mv_idx:],
        color='royalblue', linewidth=2.5, label='Efficient frontier')

# CAL
cal_x = np.linspace(0, tg_r * 1.5, 200)
cal_y = RF_MONTHLY + (tg_ret - RF_MONTHLY) / tg_r * cal_x
ax.plot(cal_x, cal_y, 'k--', linewidth=1.5, label='Capital Allocation Line')

# Individual assets
for t in TICKERS:
    ax.scatter(std_21d[t], mean_21d[t], color=COLORS[t], s=90,
               zorder=5, edgecolors='black', linewidth=0.5)
    ax.annotate(NAMES[t], (std_21d[t], mean_21d[t]),
                textcoords='offset points', xytext=(5, 2), fontsize=8)

# Portfolios
ax.scatter(mv_r, mv_ret, color='red',   s=160, marker='D', zorder=6,
           edgecolors='black', linewidth=0.5, label='Min Variance')
ax.scatter(tg_r, tg_ret, color='gold',  s=220, marker='*', zorder=6,
           edgecolors='black', linewidth=0.5, label='Tangency')
ax.scatter(0, RF_MONTHLY, color='black', s=90, marker='x', zorder=6,
           label=f'Risk-free ({RF_MONTHLY*100:.1f}%/mo)')

ax.set_xlabel('Risk — Std Dev (21-day)')
ax.set_ylabel('Expected Return (21-day)')
ax.set_title('Efficient Frontier & Capital Allocation Line\n(5-asset long-only, 21-day horizon)')
ax.legend(fontsize=8, loc='upper left')

# ── Right: Portfolio weights ─────────────────────────────────────────────
ax2  = axes[1]
x_pos = np.array([0, 1])
width = 0.14

for i, t in enumerate(TICKERS):
    wts = [mv_w[i]*100, tg_w[i]*100]
    ax2.bar(x_pos + i*width - 2*width, wts, width,
            label=NAMES[t], color=COLORS[t], edgecolor='black', linewidth=0.4)

ax2.set_xticks(x_pos)
ax2.set_xticklabels(['Min Variance\nPortfolio', 'Tangency\nPortfolio'])
ax2.set_ylabel('Weight (%)')
ax2.set_title('Portfolio Weights: Min Variance vs Tangency')
ax2.axhline(0, color='black', linewidth=0.5)
ax2.legend(fontsize=9, loc='upper right')

plt.tight_layout()
plt.savefig('figures/10_efficient_frontier.png', dpi=SAVE_DPI, bbox_inches='tight')
plt.show()
print('Saved → figures/10_efficient_frontier.png')

In [ ]:
# ── 6E. INDIVIDUAL vs PORTFOLIO RISK COMPARISON ───────────────────────────────
ind_risks   = [std_21d[t]*100 for t in TICKERS]
port_risks  = [mv_r*100, tg_r*100]
labels_all  = [NAMES[t] for t in TICKERS] + ['Min Variance\nPortfolio', 'Tangency\nPortfolio']
values_all  = ind_risks + port_risks
colors_all  = [COLORS[t] for t in TICKERS] + ['#C62828', '#FFA000']

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.bar(labels_all, values_all, color=colors_all,
              edgecolor='black', linewidth=0.5)
for b, v in zip(bars, values_all):
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.03,
            f'{v:.2f}%', ha='center', va='bottom', fontsize=9)
ax.set_ylabel('Standard Deviation (21-day, %)')
ax.set_title('Diversification Benefit: Individual Asset Risk vs Portfolio Risk (21-day Std Dev)')
ax.tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.savefig('figures/11_risk_comparison.png', dpi=SAVE_DPI, bbox_inches='tight')
plt.show()
print('Saved → figures/11_risk_comparison.png')

---
## Section 7 — Master Comparison Table

All key metrics in one place. Copy into the comparison slide.

In [ ]:
# ── 7. MASTER COMPARISON TABLE ────────────────────────────────────────────────
rows = []
for t in TICKERS:
    p    = garch_res[t]['params']
    a    = p.get('alpha[1]', 0)
    b    = p.get('beta[1]',  0)
    g    = p.get('gamma[1]', 0)
    persist = a + b + 0.5 * g

    rows.append({
        'Ticker'            : t,
        'Name'              : NAMES[t],
        'Best GARCH Model'  : garch_res[t]['model'],
        'Ann. Vol (%)'      : round(log_ret[t].std() * np.sqrt(252) * 100, 2),
        'GARCH Persistence' : round(persist, 4),
        'VaR 99% (T-dist)'  : round(var_all[t].loc[0.99, 'T-dist'], 3),
        'ES 99% (T-dist)'   : round(es_all[t].loc[0.99,  'T-dist'], 3),
    })

master = pd.DataFrame(rows).set_index('Ticker')
print('=== MASTER COMPARISON TABLE ===')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
print(master.to_string())
master.to_csv('figures/master_comparison.csv')
print('\nSaved → figures/master_comparison.csv')

# ── Key takeaways for slides ──────────────────────────────────────────────────
print('\n=== KEY TAKEAWAYS FOR SLIDES ===')
riskiest = master['Ann. Vol (%)'].idxmax()
safest   = master['Ann. Vol (%)'].idxmin()
print(f'Most volatile  (Ann. Vol): {NAMES[riskiest]} ({master.loc[riskiest, "Ann. Vol (%)"]:.1f}%)')
print(f'Least volatile (Ann. Vol): {NAMES[safest]}   ({master.loc[safest,   "Ann. Vol (%)"]:.1f}%)')
print(f'Min Variance Portfolio std (21-day): {mv_r*100:.2f}%')
print(f'Tangency Portfolio Sharpe           : {tg_sr:.3f}')
print(f'\nFull ranking by VaR 99%:')
print(master[['Name', 'Ann. Vol (%)', 'VaR 99% (T-dist)', 'ES 99% (T-dist)',
              'GARCH Persistence']].sort_values('VaR 99% (T-dist)', ascending=False).to_string())

---
## Section 8 — Out-of-Sample Evaluation: Q1 2026
> Required by guidelines ("optional: simple evaluation using data from the first trimester of 2026").
> We apply the models trained on 2023–2025 to Q1 2026 data without refitting.
> This tests whether the risk estimates were realistic forward-looking.

In [ ]:
# ── 8A. DOWNLOAD Q1 2026 DATA ─────────────────────────────────────────────────
print('Downloading Q1 2026 data…')
Q1_START, Q1_END = '2026-01-01', '2026-03-31'

px_2026 = {}
for t in TICKERS:
    print(f'  {t}…', end=' ', flush=True)
    try:
        px_2026[t] = _download_one(t, Q1_START, Q1_END)
        print(f'✓  {len(px_2026[t])} days')
    except Exception as e:
        print(f'✗  {e}')

prices_2026 = pd.DataFrame(px_2026).dropna(how='all')
log_ret_2026 = np.log(prices_2026 / prices_2026.shift(1)).dropna()

print(f'\nQ1 2026 — Period : {log_ret_2026.index[0].date()} to {log_ret_2026.index[-1].date()}')
print(f'Q1 2026 — Days   : {len(log_ret_2026)} trading-day observations')

In [ ]:
# ── 8B. VaR BACKTESTING — did Q1 2026 losses exceed the 2023-2025 VaR? ───────
# A well-calibrated 99% VaR should be breached on ~1% of days.
# In ~63 Q1 trading days → expect ~0.6 breaches. More = model underestimated risk.

print('=== VaR Backtesting: Q1 2026 out-of-sample ===')
print(f'{"Ticker":<12} {"VaR 99% (T)":>12} {"Q1 days":>8} '
      f'{"Breaches":>9} {"Breach %":>9} {"Expected":>9}')
print('-' * 62)

backtest_rows = []
for t in TICKERS:
    if t not in log_ret_2026.columns:
        continue
    # VaR from 2023-2025 model (T-dist, 99%) — already computed
    var99 = var_all[t].loc[0.99, 'T-dist']          # daily loss %
    losses_2026 = -(100 * log_ret_2026[t]).dropna()  # actual daily losses %

    n_days     = len(losses_2026)
    n_breaches = int((losses_2026 > var99).sum())
    breach_pct = n_breaches / n_days * 100 if n_days > 0 else 0
    expected   = n_days * 0.01                       # 1% of days

    flag = '⚠' if n_breaches > expected * 2 else '✓'
    print(f'{NAMES[t]:<12} {var99:>11.3f}% {n_days:>8} '
          f'{n_breaches:>9} {breach_pct:>8.1f}% {expected:>8.1f}  {flag}')

    backtest_rows.append({
        'Name'       : NAMES[t],
        'VaR 99% (%)': round(var99, 3),
        'Q1 Days'    : n_days,
        'Breaches'   : n_breaches,
        'Breach %'   : round(breach_pct, 1),
        'Expected'   : round(expected, 1),
        'Status'     : 'OVER' if n_breaches > expected * 2 else 'OK'
    })

bt_df = pd.DataFrame(backtest_rows).set_index('Name')
print('\n→ "OVER" = more breaches than expected → model underestimated Q1 2026 risk.')
print('→ "OK"   = VaR was well-calibrated for the out-of-sample period.')

In [ ]:
# ── 8C. VISUAL: EXTENDED PRICE CHART + Q1 2026 RISK COMPARISON ───────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# ── Left: Extended normalized price chart (2023–2025 + Q1 2026) ────────────
ax = axes[0]
all_prices = pd.concat([prices, prices_2026]).sort_index()
all_prices = all_prices[~all_prices.index.duplicated()]
norm_all   = all_prices.divide(all_prices.bfill().iloc[0]) * 100

for t in TICKERS:
    if t not in norm_all.columns:
        continue
    # 2023-2025 in solid line
    mask_hist = norm_all.index <= '2025-12-31'
    mask_2026 = norm_all.index >= '2026-01-01'
    ax.plot(norm_all.index[mask_hist], norm_all[t][mask_hist],
            color=COLORS[t], linewidth=1.4, label=NAMES[t])
    ax.plot(norm_all.index[mask_2026], norm_all[t][mask_2026],
            color=COLORS[t], linewidth=2.2, linestyle='--')

# Vertical line separating in-sample from out-of-sample
ax.axvline(pd.Timestamp('2026-01-01'), color='black',
           linestyle=':', linewidth=1.5, label='Q1 2026 start')
ax.set_title('Prices: 2023–2025 (solid) vs Q1 2026 (dashed)', fontweight='bold')
ax.set_ylabel('Index (Jan 2023 = 100)')
ax.legend(fontsize=8, ncol=2)

# ── Right: Q1 2026 annualised vol vs historical vol ───────────────────────
ax2 = axes[1]
hist_vols = [log_ret[t].std() * np.sqrt(252) * 100 for t in TICKERS
             if t in log_ret.columns]
q1_vols   = [log_ret_2026[t].std() * np.sqrt(252) * 100
             if t in log_ret_2026.columns else 0 for t in TICKERS]
labels    = [NAMES[t] for t in TICKERS]
x = np.arange(len(labels))
w = 0.35

bars1 = ax2.bar(x - w/2, hist_vols, w, label='2023–2025 (in-sample)',
                color=[COLORS[t] for t in TICKERS], alpha=0.6,
                edgecolor='black', linewidth=0.5)
bars2 = ax2.bar(x + w/2, q1_vols,   w, label='Q1 2026 (out-of-sample)',
                color=[COLORS[t] for t in TICKERS], alpha=1.0,
                edgecolor='black', linewidth=0.5, hatch='//')

ax2.set_xticks(x)
ax2.set_xticklabels(labels, rotation=15)
ax2.set_ylabel('Annualised Volatility (%)')
ax2.set_title('Volatility: Historical vs Q1 2026', fontweight='bold')
ax2.legend(fontsize=9)

plt.tight_layout()
plt.savefig('figures/12_q1_2026_evaluation.png', dpi=SAVE_DPI, bbox_inches='tight')
plt.show()
print('Saved → figures/12_q1_2026_evaluation.png')

# Summary
print('\n=== Q1 2026 VOLATILITY vs HISTORICAL ===')
for t, hv, qv in zip(TICKERS, hist_vols, q1_vols):
    change = qv - hv
    direction = '↑ HIGHER' if change > 2 else ('↓ lower' if change < -2 else '≈ similar')
    print(f'{NAMES[t]:20s}  hist={hv:.1f}%  Q1={qv:.1f}%  {direction}')

---
## Generate PPTX Presentation
Run this cell to produce `Financial_Analytics_Presentation.pptx`. Download it from the Files panel (left sidebar).